# Holiday Calendar (Idiomatic Notebook)

This notebook uses the well-supported `holidays` package to generate holiday dates with a concise, notebook-first workflow.

In [1]:
# Notebook parameters
start_year = 2024
end_year = 2028
country = "US"
subdiv = None  # Example: "CA" for California

In [2]:
# Install once per environment if needed
%pip -q install holidays

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import pandas as pd
import holidays

In [4]:
# Build a tidy holiday table
years = range(start_year, end_year + 1)
cal = holidays.country_holidays(country, subdiv=subdiv, years=years, observed=True)

df = (
    pd.Series(cal, name="Holiday")
    .rename_axis("Date")
    .sort_index()
    .reset_index()
)

df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Weekday"] = df["Date"].dt.day_name()

df.head(10)

,Date,Holiday,Year,Weekday
0,2024-01-01,New Year's Day,2024,Monday
1,2024-01-15,Martin Luther King Jr. Day,2024,Monday
2,2024-02-19,Washington's Birthday,2024,Monday
3,2024-05-27,Memorial Day,2024,Monday
4,2024-06-19,Juneteenth National Independence Day,2024,Wednesday
5,2024-07-04,Independence Day,2024,Thursday
6,2024-09-02,Labor Day,2024,Monday
7,2024-10-14,Columbus Day,2024,Monday
8,2024-11-11,Veterans Day,2024,Monday
9,2024-11-28,Thanksgiving Day,2024,Thursday


In [5]:
# Compact yearly summary
summary = (
    df.groupby("Year", as_index=False)
    .agg(Holiday_Count=("Holiday", "count"))
)
summary

,Year,Holiday_Count
0,2024,11
1,2025,11
2,2026,12
3,2027,15
4,2028,12


In [6]:
# Inspect one year
year = 2026
df.loc[df["Year"] == year, ["Date", "Weekday", "Holiday"]].reset_index(drop=True)

,Date,Weekday,Holiday
0,2026-01-01,Thursday,New Year's Day
1,2026-01-19,Monday,Martin Luther King Jr. Day
2,2026-02-16,Monday,Washington's Birthday
3,2026-05-25,Monday,Memorial Day
4,2026-06-19,Friday,Juneteenth National Independence Day
5,2026-07-03,Friday,Independence Day (observed)
6,2026-07-04,Saturday,Independence Day
7,2026-09-07,Monday,Labor Day
8,2026-10-12,Monday,Columbus Day
9,2026-11-11,Wednesday,Veterans Day


## Quick Validation Tests

Each test is a single code cell that shows expected vs actual values as notebook output.
Assertions enforce correctness without print-based status messages.

In [7]:
# JSON-LD output type and helpers (type-driven MIME output)
from pandas.testing import assert_frame_equal

class JsonLDResult:
    def __init__(self, payload, metadata=None):
        self.payload = payload
        self.metadata = metadata or {}

    def _repr_mimebundle_(self, include=None, exclude=None):
        data = {
            "application/ld+json": self.payload,
            "application/json": self.payload,
            "text/plain": json.dumps(self.payload, indent=2),
        }
        metadata = {
            "application/json": {"expanded": True},
            "application/ld+json": {"expanded": True},
        }
        metadata.update(self.metadata)
        return data, metadata

def to_iso_records(frame):
    normalized = frame.copy()
    for col in normalized.columns:
        if pd.api.types.is_datetime64_any_dtype(normalized[col]):
            normalized[col] = normalized[col].dt.strftime("%Y-%m-%d")
    return normalized.to_dict("records")

def jsonld_test_result(test_name, expected, actual, passed):
    return JsonLDResult({
        "@context": {
            "@vocab": "https://schema.org/",
            "testName": "name",
            "expected": "expectedValue",
            "actual": "value",
            "passed": "result",
        },
        "@type": "PropertyValue",
        "testName": test_name,
        "expected": expected,
        "actual": actual,
        "passed": passed,
    })

def jsonld_approval_result(test_name, approved_df, actual_df, sort_by):
    approved = approved_df.sort_values(sort_by).reset_index(drop=True)
    actual = actual_df.sort_values(sort_by).reset_index(drop=True)

    try:
        assert_frame_equal(actual, approved, check_dtype=False)
        passed = True
        diff_records = []
    except AssertionError:
        passed = False
        left = approved.assign(_source="Approved")
        right = actual.assign(_source="Actual")
        diff = pd.concat([left, right], ignore_index=True).drop_duplicates(keep=False)
        diff_records = to_iso_records(diff)

    payload = {
        "@context": {
            "@vocab": "https://schema.org/",
            "testName": "name",
            "approved": "expectedValue",
            "actual": "value",
            "passed": "result",
            "diff": "disambiguatingDescription",
        },
        "@type": "PropertyValue",
        "testName": test_name,
        "approved": to_iso_records(approved),
        "actual": to_iso_records(actual),
        "passed": passed,
        "diff": diff_records,
    }
    assert passed, f"{test_name} failed. Diff: {json.dumps(diff_records, ensure_ascii=True)}"
    return JsonLDResult(payload)

In [8]:
# Test 1: required columns are present (one-cell test)
expected_columns = sorted(["Date", "Holiday", "Year", "Weekday"])
actual_columns = sorted(df.columns.tolist())
missing = [c for c in expected_columns if c not in actual_columns]
assert not missing, f"Missing columns: {missing}"

jsonld_test_result(
    "required_columns_present",
    {"columns": expected_columns},
    {"columns": actual_columns},
    True,
 )

{
  "@context": {
    "@vocab": "https://schema.org/",
    "testName": "name",
    "expected": "expectedValue",
    "actual": "value",
    "passed": "result"
  },
  "@type": "PropertyValue",
  "testName": "required_columns_present",
  "expected": {
    "columns": [
      "Date",
      "Holiday",
      "Weekday",
      "Year"
    ]
  },
  "actual": {
    "columns": [
      "Date",
      "Holiday",
      "Weekday",
      "Year"
    ]
  },
  "passed": true
}

In [9]:
# Test 2: year boundaries are correct (one-cell test)
actual_min = int(df["Year"].min())
actual_max = int(df["Year"].max())
assert actual_min == start_year, f"Start year mismatch: expected {start_year}, got {actual_min}"
assert actual_max == end_year, f"End year mismatch: expected {end_year}, got {actual_max}"

jsonld_test_result(
    "year_boundaries",
    {"startYear": start_year, "endYear": end_year},
    {"startYear": actual_min, "endYear": actual_max},
    True,
 )

{
  "@context": {
    "@vocab": "https://schema.org/",
    "testName": "name",
    "expected": "expectedValue",
    "actual": "value",
    "passed": "result"
  },
  "@type": "PropertyValue",
  "testName": "year_boundaries",
  "expected": {
    "startYear": 2024,
    "endYear": 2028
  },
  "actual": {
    "startYear": 2024,
    "endYear": 2028
  },
  "passed": true
}

In [10]:
# Test 3: date ordering and weekday derivation (one-cell test)
is_sorted = bool(df["Date"].is_monotonic_increasing)
weekday_match = bool((df["Date"].dt.day_name() == df["Weekday"]).all())
assert is_sorted, "Dates are not sorted ascending"
assert weekday_match, "Weekday values do not match Date"

jsonld_test_result(
    "date_order_and_weekday_derivation",
    {"datesSorted": True, "weekdayMatchesDate": True},
    {"datesSorted": is_sorted, "weekdayMatchesDate": weekday_match},
    True,
 )

{
  "@context": {
    "@vocab": "https://schema.org/",
    "testName": "name",
    "expected": "expectedValue",
    "actual": "value",
    "passed": "result"
  },
  "@type": "PropertyValue",
  "testName": "date_order_and_weekday_derivation",
  "expected": {
    "datesSorted": true,
    "weekdayMatchesDate": true
  },
  "actual": {
    "datesSorted": true,
    "weekdayMatchesDate": true
  },
  "passed": true
}

In [11]:
# Test 4: known holiday spot checks (one-cell test)
known = pd.DataFrame([
    {"Date": pd.Timestamp("2024-01-01"), "Expected": "New Year's Day"},
    {"Date": pd.Timestamp("2024-11-28"), "Expected": "Thanksgiving Day"},
    {"Date": pd.Timestamp("2026-05-25"), "Expected": "Memorial Day"},
])

actual = (
    df.merge(known[["Date"]], on="Date", how="inner")
      .groupby("Date", as_index=False)
      .agg(Actual=("Holiday", lambda s: " | ".join(sorted(set(s)))))
)

comparison = known.merge(actual, on="Date", how="left")
comparison["Pass"] = comparison.apply(
    lambda r: isinstance(r["Actual"], str) and r["Expected"] in r["Actual"], axis=1
)
assert comparison["Pass"].all(), "Known holiday spot check failed"

jsonld_test_result(
    "known_holiday_spot_checks",
    to_iso_records(known),
    to_iso_records(comparison),
    True,
 )

{
  "@context": {
    "@vocab": "https://schema.org/",
    "testName": "name",
    "expected": "expectedValue",
    "actual": "value",
    "passed": "result"
  },
  "@type": "PropertyValue",
  "testName": "known_holiday_spot_checks",
  "expected": [
    {
      "Date": "2024-01-01",
      "Expected": "New Year's Day"
    },
    {
      "Date": "2024-11-28",
      "Expected": "Thanksgiving Day"
    },
    {
      "Date": "2026-05-25",
      "Expected": "Memorial Day"
    }
  ],
  "actual": [
    {
      "Date": "2024-01-01",
      "Expected": "New Year's Day",
      "Actual": "New Year's Day",
      "Pass": true
    },
    {
      "Date": "2024-11-28",
      "Expected": "Thanksgiving Day",
      "Actual": "Thanksgiving Day",
      "Pass": true
    },
    {
      "Date": "2026-05-25",
      "Expected": "Memorial Day",
      "Actual": "Memorial Day",
      "Pass": true
    }
  ],
  "passed": true
}

## Concise Approval Tests (Visible Approved vs Actual)

This style keeps approvals directly in notebook cells.
Each test cell displays Approved and Actual values, then fails with a focused diff when they differ.

### Notebook Testing Patterns In Practice

- ipytest: run pytest directly inside notebook cells via magics.
- nbval or pytest-notebook: rerun whole notebooks and compare stored outputs, usually in CI.
- testbook: write tests in separate Python files against notebook code.
- ApprovalTests.Python: classic approved vs received workflow with diff tools.

This notebook uses per-cell approved tables so each test visibly shows Approved and Actual values in one place.

In [12]:
# Helper note: JSON-LD helpers are defined above the tests to support linear top-to-bottom execution.
# This cell is intentionally left minimal to avoid redefining hooks later in the notebook.

In [13]:
# Approval test 1: full 2026 holiday list (explicit approved values)
approved_2026 = pd.DataFrame([
    {"Date": "2026-01-01", "Holiday": "New Year's Day"},
    {"Date": "2026-01-19", "Holiday": "Martin Luther King Jr. Day"},
    {"Date": "2026-02-16", "Holiday": "Washington's Birthday"},
    {"Date": "2026-05-25", "Holiday": "Memorial Day"},
    {"Date": "2026-06-19", "Holiday": "Juneteenth National Independence Day"},
    {"Date": "2026-07-03", "Holiday": "Independence Day (observed)"},
    {"Date": "2026-07-04", "Holiday": "Independence Day"},
    {"Date": "2026-09-07", "Holiday": "Labor Day"},
    {"Date": "2026-10-12", "Holiday": "Columbus Day"},
    {"Date": "2026-11-11", "Holiday": "Veterans Day"},
    {"Date": "2026-11-26", "Holiday": "Thanksgiving Day"},
    {"Date": "2026-12-25", "Holiday": "Christmas Day"},
])
approved_2026["Date"] = pd.to_datetime(approved_2026["Date"])

actual_2026 = df.loc[df["Year"] == 2026, ["Date", "Holiday"]]
jsonld_approval_result(
    "approval_us_2026_holidays",
    approved_2026,
    actual_2026,
    sort_by=["Date", "Holiday"],
 )

{
  "@context": {
    "@vocab": "https://schema.org/",
    "testName": "name",
    "approved": "expectedValue",
    "actual": "value",
    "passed": "result",
    "diff": "disambiguatingDescription"
  },
  "@type": "PropertyValue",
  "testName": "approval_us_2026_holidays",
  "approved": [
    {
      "Date": "2026-01-01",
      "Holiday": "New Year's Day"
    },
    {
      "Date": "2026-01-19",
      "Holiday": "Martin Luther King Jr. Day"
    },
    {
      "Date": "2026-02-16",
      "Holiday": "Washington's Birthday"
    },
    {
      "Date": "2026-05-25",
      "Holiday": "Memorial Day"
    },
    {
      "Date": "2026-06-19",
      "Holiday": "Juneteenth National Independence Day"
    },
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    },
    {
      "Date": "2026-09-07",
      "Holiday": "Labor Day"
    },
    {
      "Date": "2026-10-12",
      "Holiday": "Columbus

In [14]:
# Approval test 2: Independence Day observed behavior in 2026
approved_observed = pd.DataFrame([
    {"Date": "2026-07-03", "Holiday": "Independence Day (observed)"},
    {"Date": "2026-07-04", "Holiday": "Independence Day"},
])
approved_observed["Date"] = pd.to_datetime(approved_observed["Date"])

actual_observed = df.loc[
    (df["Date"].between(pd.Timestamp("2026-07-03"), pd.Timestamp("2026-07-04")))
    & (df["Holiday"].str.contains("Independence Day")),
    ["Date", "Holiday"],
]
jsonld_approval_result(
    "approval_independence_day_observed_2026",
    approved_observed,
    actual_observed,
    sort_by=["Date", "Holiday"],
 )

{
  "@context": {
    "@vocab": "https://schema.org/",
    "testName": "name",
    "approved": "expectedValue",
    "actual": "value",
    "passed": "result",
    "diff": "disambiguatingDescription"
  },
  "@type": "PropertyValue",
  "testName": "approval_independence_day_observed_2026",
  "approved": [
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    }
  ],
  "actual": [
    {
      "Date": "2026-07-03",
      "Holiday": "Independence Day (observed)"
    },
    {
      "Date": "2026-07-04",
      "Holiday": "Independence Day"
    }
  ],
  "passed": true,
  "diff": []
}